In [ ]:
!pip install -q shapely


In [ ]:
import os, sys, json
import numpy as np, pandas as pd
from google.colab import drive
drive.mount("/content/drive")

ORDNER = "/content/drive/MyDrive/llm_astar_max"
pfad = f"{ORDNER}/table1_reproduction"
sys.path.insert(0, ORDNER + "/llm-astar")
from llmastar.utils import list_parse

PROMPTS = ["standard", "cot", "repe"]
antworten = {pt: json.load(open(f"{pfad}/antworten_{pt}.json")) for pt in PROMPTS}
df = pd.DataFrame(json.load(open(f"{pfad}/ergebnisse.json")))

def wegpunkte(antwort):
    p = list_parse(antwort or "")
    return p if isinstance(p, list) else []

for pt in PROMPTS:
    anzahl = [len(wegpunkte(antworten[pt][sid])) for sid in df.id]
    ohne = sum(1 for n in anzahl if n <= 2)
    ops = df[f"{pt}_op"] / df.astar_op
    print(f"{pt:8s} | Ø Wegpunkte: {np.mean(anzahl):.1f}"
          f" | Antworten ohne Zwischenpunkte: {ohne}"
          f" | schlechter als A*: {int((ops > 1).sum())}"
          f" | Median Operation %: {100 * ops.median():.1f}")


Mounted at /content/drive
standard | Ø Wegpunkte: 10.0 | Antworten ohne Zwischenpunkte: 83 | schlechter als A*: 173 | Median Operation %: 47.2
cot      | Ø Wegpunkte: 7.4 | Antworten ohne Zwischenpunkte: 28 | schlechter als A*: 172 | Median Operation %: 49.6
repe     | Ø Wegpunkte: 5.1 | Antworten ohne Zwischenpunkte: 71 | schlechter als A*: 192 | Median Operation %: 53.0


In [ ]:
import math, copy, io, contextlib
from llmastar.env.search import plotting
plotting.Plotting.animation = lambda self, *args, **kwargs: None
from llmastar.pather import AStar
from llmastar.utils import is_lines_collision

karten = json.load(open(ORDNER + "/llm-astar/dataset/environment_50_30.json"))
samples = []
for karte in karten:
    for k, (start, ziel, text) in enumerate(karte["start_goal"]):
        samples.append({"id": f"{karte['id']}_{k}",
            "query": {"start": start, "goal": ziel, "size": [51, 31],
                      "horizontal_barriers": karte["horizontal_barriers"],
                      "vertical_barriers": karte["vertical_barriers"],
                      "range_x": karte["range_x"], "range_y": karte["range_y"]}})
samples = [s for s in samples if s["id"] in set(df.id)]

def leise(funktion):
    with contextlib.redirect_stdout(io.StringIO()):
        return funktion()

def gm(x):
    return 100 * np.exp(np.log(np.asarray(x, dtype=float)).mean())

print(len(samples), "Beispiele geladen")   # sollte 1000 sein


1000 Beispiele geladen


In [ ]:
for pt in PROMPTS:
    mit = np.array([len(wegpunkte(antworten[pt][sid])) > 2 for sid in df.id])
    ops = (df[f"{pt}_op"] / df.astar_op).values
    lae = (df[f"{pt}_len"] / df.astar_len).values
    zeile = f"{pt:8s} | MIT Zwischenpunkten: {mit.sum():4d} Beispiele, Op {gm(ops[mit]):5.1f} %, Länge {gm(lae[mit]):6.2f} %"
    if (~mit).any():
        zeile += f" | OHNE: {(~mit).sum():4d} Beispiele, Op {gm(ops[~mit]):5.1f} %, Länge {gm(lae[~mit]):6.2f} %"
    print(zeile)


standard | MIT Zwischenpunkten:  917 Beispiele, Op  47.4 %, Länge 102.83 % | OHNE:   83 Beispiele, Op  33.6 %, Länge 101.71 %
cot      | MIT Zwischenpunkten:  972 Beispiele, Op  48.1 %, Länge 102.65 % | OHNE:   28 Beispiele, Op  33.1 %, Länge 101.13 %
repe     | MIT Zwischenpunkten:  929 Beispiele, Op  52.4 %, Länge 102.35 % | OHNE:   71 Beispiele, Op  28.3 %, Länge 101.57 %


In [ ]:
class GewichteterAStar(AStar):
    def __init__(self, eps, dynamisch=True):
        super().__init__()
        self.eps, self.dynamisch = eps, dynamisch
    def f_value(self, s):
        if self.dynamisch:
            N = math.dist(self.s_start, self.s_goal)
            w = 1 + self.eps * max(0, 1 - self.g[s] / N)
        else:
            w = 1 + self.eps
        return self.g[s] + w * self.heuristic(s)

a = df.set_index("id")
varianten = [("Dynamisch, eps=0.25", 0.25, True), ("Dynamisch, eps=0.5", 0.5, True),
             ("Dynamisch, eps=1.0", 1.0, True), ("Fest, w=2 (ohne LLM)", 1.0, False)]
for name, eps, dyn in varianten:
    o, st, l = [], [], []
    for s in samples:
        r = leise(lambda: GewichteterAStar(eps, dyn).searching(query=copy.deepcopy(s["query"])))
        b = a.loc[s["id"]]
        o.append(r["operation"] / b.astar_op)
        st.append(r["storage"] / b.astar_st)
        l.append(r["length"] / b.astar_len)
    print(f"{name:22s} | Op {gm(o):6.2f} % | Storage {gm(st):6.2f} % | Länge {gm(l):6.2f} %")


Dynamisch, eps=0.25    | Op  54.66 % | Storage  69.94 % | Länge 100.28 %
Dynamisch, eps=0.5     | Op  43.76 % | Storage  63.08 % | Länge 101.53 %
Dynamisch, eps=1.0     | Op  35.90 % | Storage  58.49 % | Länge 104.12 %
Fest, w=2 (ohne LLM)   | Op  34.32 % | Storage  56.23 % | Länge 102.02 %
